# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator
import re
from concurrent.futures import ThreadPoolExecutor


---
# Dataset 1: USDA
## Exploration

In [2]:
usda_df = pd.read_csv('../data/bronze_layer/usda.csv')

In [3]:
usda_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fdc_id             40000 non-null  int64  
 1   food_name          40000 non-null  str    
 2   data_type          40000 non-null  str    
 3   food_category      39954 non-null  str    
 4   brand_owner        31383 non-null  str    
 5   brand_name         30254 non-null  str    
 6   ingredients        31656 non-null  str    
 7   serving_size       31842 non-null  float64
 8   serving_unit       31842 non-null  str    
 9   household_serving  30790 non-null  str    
 10  calories           39538 non-null  float64
 11  carbs_g            39411 non-null  float64
 12  calcium_mg         34215 non-null  float64
 13  fat_g              39713 non-null  float64
 14  protein_g          39747 non-null  float64
 15  saturated_fat_g    35014 non-null  float64
 16  vitamin_c_mg       22940 non-null

In [4]:
usda_df.isnull().sum()

fdc_id                   0
food_name                0
data_type                0
food_category           46
brand_owner           8617
brand_name            9746
ingredients           8344
serving_size          8158
serving_unit          8158
household_serving     9210
calories               462
carbs_g                589
calcium_mg            5785
fat_g                  287
protein_g              253
saturated_fat_g       4986
vitamin_c_mg         17060
fiber_g               6275
iron_mg               5750
sodium_mg              617
sugar_g               4261
cholesterol_mg        5842
health_score             0
food_type                0
dtype: int64

In [5]:
usda_df.describe()

,fdc_id,serving_size,calories,carbs_g,calcium_mg,fat_g,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score
count,4.000000e+04,31842.000000,39538.000000,39411.000000,34215.000000,39713.000000,39747.000000,35014.000000,22940.000000,33725.000000,34250.000000,3.938300e+04,35739.000000,34158.000000,40000.000000
mean,1.793461e+06,92.563962,308.159861,28.420500,106.091172,12.118343,8.260691,4.451546,24.138259,2.568491,2.389289,1.067233e+03,13.262690,62.575833,48.887250
std,8.964022e+05,100.581765,392.575965,47.614576,1327.632746,19.140351,10.668951,7.387522,1419.157682,5.662308,60.647733,2.895701e+04,36.323827,2141.337343,9.850912
min,1.675120e+05,0.000000,0.000000,-0.705000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,20.000000
25%,1.459896e+06,28.000000,85.000000,3.570000,5.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.030000,1.600000e+01,0.000000,0.000000,40.000000
50%,2.094756e+06,55.000000,250.000000,15.600000,29.000000,4.550000,5.130000,1.670000,0.000000,1.000000,0.980000,1.430000e+02,4.520000,0.000000,50.000000
75%,2.469605e+06,118.000000,404.000000,51.800000,98.000000,17.600000,12.500000,6.077500,5.200000,3.300000,2.280000,4.810000e+02,16.700000,38.000000,55.000000
max,2.752382e+06,5055.000000,37600.000000,6160.000000,150000.000000,1170.000000,760.000000,300.000000,210000.000000,588.000000,8930.000000,4.690000e+06,5560.000000,214000.000000,75.000000


In [6]:
(usda_df['carbs_g'] < 0).sum()

np.int64(10)

In [7]:
pd.set_option('display.max_columns', None)
usda_df[usda_df['carbs_g'] < 0]

,fdc_id,food_name,data_type,food_category,brand_owner,brand_name,ingredients,serving_size,serving_unit,household_serving,calories,carbs_g,calcium_mg,fat_g,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score,food_type
1911,2727571,"Bison, ground, raw",Foundation,"Lamb, Veal, and Game Products",NaN,NaN,NaN,NaN,NaN,NaN,164.0,-0.1500,6.83,8.880,19.9,NaN,NaN,NaN,2.170,55.7,NaN,64.5,60,Other
2743,2727569,"Chicken, breast, meat and skin, raw",Foundation,Poultry Products,NaN,NaN,NaN,NaN,NaN,NaN,133.0,-0.4280,6.90,4.780,21.4,NaN,NaN,NaN,0.354,48.1,NaN,75.4,60,Meat & Poultry
2903,2727566,"Chicken, drumstick, meat and skin, raw",Foundation,Poultry Products,NaN,NaN,NaN,NaN,NaN,NaN,130.0,-0.4750,8.28,5.940,18.4,NaN,NaN,NaN,0.666,91.0,NaN,94.6,60,Meat & Poultry
2949,2727567,"Chicken, thigh, meat and skin, raw",Foundation,Poultry Products,NaN,NaN,NaN,NaN,NaN,NaN,193.0,-0.1730,5.73,13.400,17.1,NaN,NaN,NaN,0.586,63.6,NaN,95.5,60,Meat & Poultry
2954,2727568,"Chicken, wing, meat and skin, raw",Foundation,Poultry Products,NaN,NaN,NaN,NaN,NaN,NaN,173.0,-0.4590,13.60,10.600,18.4,NaN,NaN,NaN,0.511,84.1,NaN,98.9,60,Meat & Poultry
4166,2747656,"Halibut, frozen, wild caught",Foundation,Finfish and Shellfish Products,NaN,NaN,NaN,NaN,NaN,NaN,81.3,-0.0602,4.10,0.592,19.1,NaN,NaN,NaN,0.000,108.0,NaN,NaN,60,Other
4540,2727570,"Lamb, ground, raw",Foundation,"Lamb, Veal, and Game Products",NaN,NaN,NaN,NaN,NaN,NaN,242.0,-0.2510,6.58,18.600,17.5,NaN,NaN,NaN,1.640,53.4,NaN,75.0,60,Other
5904,2727576,"Pork, belly, with skin, raw",Foundation,Pork Products,NaN,NaN,NaN,NaN,NaN,NaN,385.0,-0.7050,4.20,35.800,15.2,NaN,NaN,NaN,0.382,49.7,NaN,66.6,60,Meat & Poultry
5905,2727575,"Pork, chop, center cut, raw",Foundation,Pork Products,NaN,NaN,NaN,NaN,NaN,NaN,145.0,-0.5620,4.12,5.480,22.8,NaN,NaN,NaN,0.421,39.3,NaN,57.3,60,Meat & Poultry
7675,2747673,"Tuna, ahi or yellowfin, frozen, wild caught",Foundation,Finfish and Shellfish Products,NaN,NaN,NaN,NaN,NaN,NaN,102.0,-0.1040,3.19,0.388,24.7,NaN,NaN,NaN,0.591,94.4,NaN,NaN,60,Other


In [8]:
usda_df[usda_df['food_category'].isnull()]

,fdc_id,food_name,data_type,food_category,brand_owner,brand_name,ingredients,serving_size,serving_unit,household_serving,calories,carbs_g,calcium_mg,fat_g,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score,food_type
8486,767542,.5-1 LB IPW CLEAN OCTOPUS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,Octopus.,112.0,g,4 OZ,80.0,2.68,54.0,0.89,15.20,0.00,1.1,0.0,4.46,232.0,0.00,45.0,60,Other
9050,768224,1-2 IPW CLEANED OCTOPUS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,Octopus.,112.0,g,4 OZ,80.0,2.68,54.0,0.89,15.20,0.00,1.6,0.0,4.46,232.0,0.00,45.0,60,Other
9109,766348,1/2 IPW CLEANED OCTOPUS,Branded,NaN,"BEAVER STREET FISHERIES, INC.",NaN,Octopus.,113.0,g,4 oz,97.0,2.65,44.0,1.77,15.00,0.00,1.6,0.0,0.66,44.0,0.00,230.0,60,Other
9292,763256,10/20 CT IQF SCALLOPS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Scallops, water, sodium tripolyphosphate (to r...",113.0,g,4 OZ,111.0,1.77,23.0,0.88,18.60,0.00,1.6,0.0,0.64,230.0,0.00,49.0,60,Other
9293,761604,10/20 CT IQF SCALLOPS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Scallops, water, sodium tripolyphosphate (to r...",112.0,g,4 oz,71.0,2.68,6.0,0.45,12.50,0.00,0.0,0.0,0.38,393.0,0.00,22.0,60,Other
9294,763338,10/20 CT IQF SCALLOPS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Scallops, water, sodium tripolyphosphate.",112.0,g,4 oz,71.0,3.57,6.0,0.45,12.50,0.00,0.0,0.0,0.38,393.0,0.00,22.0,60,Other
9295,765122,10/20 CT IQF SCALLOPS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Scallops, water, sodium tripolyphosphate (to r...",112.0,g,4 oz,71.0,2.68,6.0,0.45,12.50,0.00,0.0,0.0,0.38,393.0,0.00,22.0,60,Other
9296,768652,10/20 CT IQF SCALLOPS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Scallops, water, sodium tripolyphosphate (to r...",112.0,g,4 oz,71.0,2.68,6.0,0.45,12.50,0.00,0.0,0.0,0.38,393.0,0.00,22.0,60,Other
13362,765780,16/25 CT BABY OCTOPUS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,"Octopus, salt.",112.0,g,4 oz,62.0,1.79,80.0,0.89,11.60,0.00,0.0,0.0,0.67,188.0,0.00,36.0,60,Other
14164,768104,2-4 IPW CLEANED OCTOPUS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,Octopus.,112.0,g,4 oz,80.0,2.68,46.0,0.89,15.20,0.00,1.6,0.0,4.46,232.0,0.00,45.0,60,Other


In [9]:
usda_df[usda_df['brand_owner'] == 'Beaver Street Fisheries Inc.']

,fdc_id,food_name,data_type,food_category,brand_owner,brand_name,ingredients,serving_size,serving_unit,household_serving,calories,carbs_g,calcium_mg,fat_g,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score,food_type
8484,1458972,.5-.75 WGGS LANE SNAPPER,Branded,Fish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,PACKERS,Snapper.,112.0,g,4 oz,98.0,0.00,23.0,1.34,20.5,0.00,1.6,0.0,0.18,62.0,0.00,36.0,60,Other
8485,1458715,.5-.75 WGGS LANE SNAPPER,Branded,Fish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,PACKERS,Lane Snapper,112.0,g,4 oz,98.0,0.00,32.0,1.34,20.5,0.00,1.6,0.0,0.18,62.0,0.00,36.0,60,Other
8486,767542,.5-1 LB IPW CLEAN OCTOPUS,Branded,NaN,Beaver Street Fisheries Inc.,NaN,Octopus.,112.0,g,4 OZ,80.0,2.68,54.0,0.89,15.2,0.00,1.1,0.0,4.46,232.0,0.00,45.0,60,Other
8487,1459793,.5/1 LB W.G.G.S. SNAPPER,Branded,Fish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,PACKERS,Snapper,112.0,g,4 oz,98.0,0.00,23.0,1.34,20.5,0.00,1.6,0.0,0.18,62.0,0.00,36.0,60,Other
8489,1457985,.75-1 LB BLUERUNNERS,Branded,Fish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,PACKERS,Bluerunners,112.0,g,4 oz,107.0,0.00,4.0,1.79,21.4,0.00,0.0,0.0,0.00,36.0,0.00,45.0,60,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34667,1459621,BAH. CLEANED CONCH,Branded,Shellfish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,PACKERS,Conch,112.0,g,4 oz,89.0,2.68,27.0,1.79,17.0,0.00,1.6,0.0,0.45,214.0,0.00,31.0,60,Other
35795,1458115,BALLYHOO,Branded,Fish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,JUST RITE,not for human consumption,113.0,g,4 oz,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.0,50,Other
37242,1460371,BANG'N BREADED SHRIMP,Branded,Shellfish Unprepared/Unprocessed,Beaver Street Fisheries Inc.,BANG'N SHRIMP,"SHRIMP, BEER (WATER, BARLEY, MALT, HOPS, RICE,...",113.0,g,4 OZ,186.0,30.10,23.0,0.88,13.3,0.00,0.0,0.9,2.23,310.0,1.77,88.0,60,Grains
38724,1458023,BASEBALL CUT CH SIRLOIN,Branded,Meat/Poultry/Other Animals Unprepared/Unproce...,Beaver Street Fisheries Inc.,H.F.'S,Baseball cut Sirloin,112.0,g,4 oz,214.0,0.00,25.0,14.30,19.6,5.36,0.0,0.0,1.43,54.0,0.00,80.0,50,Meat & Poultry


# Data Cleaning

**Steps:**
- Check for duplicate rows
- Drop columns not needed for analysis: `household_serving`, `serving_size`, `serving_unit`, `ingredients`
- Fill missing `food_category` values with `'Fish Unprepared/Unprocessed'` (identified via domain inspection of Beaver Street Fisheries rows)
- Fill missing `brand_owner` / `brand_name` using tiered logic:
  - SR Legacy & Foundation rows → `'No Brand'`
  - Branded rows → cross-fill from the other brand column when one is present
  - Remaining nulls → mode per `food_category`
- Replace negative `carbs_g` values with 0
- **Fix calorie outliers:** rows where `calories ≈ 4× expected_from_macros` indicate a per-serving recording error; recalculate calories from fat/protein/carbs instead of clamping
- **Fix sodium outliers:** values > 10,000 mg/100g (or > 50,000 for seasonings) are data-entry errors; set to NaN and fill with grouped median
- Fill remaining nulls in numeric columns using median grouped by `food_type`; global median as fallback
- **Clean `food_name`:** lowercase, remove punctuation, collapse whitespace; drop rows with no real word content (< 2 letters)
- **Fix brand names:** replace numeric-only / barcode-like strings in `brand_owner` and `brand_name` with `'Unknown'`; strip stray brackets from brand strings


In [10]:
usda_df.duplicated().sum()


np.int64(0)

In [11]:
# Drop columns not needed for analysis
usda_df = usda_df.drop(columns=['household_serving', 'serving_size', 'serving_unit', 'ingredients'])
usda_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fdc_id           40000 non-null  int64  
 1   food_name        40000 non-null  str    
 2   data_type        40000 non-null  str    
 3   food_category    39954 non-null  str    
 4   brand_owner      31383 non-null  str    
 5   brand_name       30254 non-null  str    
 6   calories         39538 non-null  float64
 7   carbs_g          39411 non-null  float64
 8   calcium_mg       34215 non-null  float64
 9   fat_g            39713 non-null  float64
 10  protein_g        39747 non-null  float64
 11  saturated_fat_g  35014 non-null  float64
 12  vitamin_c_mg     22940 non-null  float64
 13  fiber_g          33725 non-null  float64
 14  iron_mg          34250 non-null  float64
 15  sodium_mg        39383 non-null  float64
 16  sugar_g          35739 non-null  float64
 17  cholesterol_mg   34158 

In [12]:
# Fill missing food_category (manual inspection: Beaver Street Fisheries)
usda_df['food_category'] = usda_df['food_category'].fillna('Fish Unprepared/Unprocessed')

In [13]:
# ── SR Legacy & Foundation rows naturally have no brand ───────────────
no_brand_mask = usda_df['data_type'].isin(['SR Legacy', 'Foundation'])
usda_df.loc[no_brand_mask, 'brand_owner'] = usda_df.loc[no_brand_mask, 'brand_owner'].fillna('No Brand')
usda_df.loc[no_brand_mask, 'brand_name']  = usda_df.loc[no_brand_mask, 'brand_name'].fillna('No Brand')

# ── Branded rows – cross-fill ─────────────────────────────────────────
branded_mask = usda_df['data_type'] == 'Branded'

usda_df.loc[branded_mask, 'brand_owner'] = usda_df.loc[branded_mask].apply(
    lambda row: row['brand_name']
    if pd.isnull(row['brand_owner']) and pd.notnull(row['brand_name'])
    else row['brand_owner'], axis=1
)
usda_df.loc[branded_mask, 'brand_name'] = usda_df.loc[branded_mask].apply(
    lambda row: row['brand_owner']
    if pd.isnull(row['brand_name']) and pd.notnull(row['brand_owner'])
    else row['brand_name'], axis=1
)

# ── Fill remaining nulls using category mode ──────────────────────────
for col in ['brand_owner', 'brand_name']:
    usda_df.loc[branded_mask, col] = (
        usda_df.loc[branded_mask]
        .groupby('food_category')[col]
        .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))
    )

print(usda_df[['brand_owner', 'brand_name']].isnull().sum())


brand_owner    0
brand_name     0
dtype: int64


In [14]:
# ── Fix numeric-only brand values ─────────────────────────────────────
# Patterns: all digits, barcode-like strings (long digit sequences), date-like strings
numeric_brand_pattern = r'^[\d\s/\.\-]+$'

usda_df['brand_owner'] = usda_df['brand_owner'].apply(
    lambda x: 'Unknown' if re.fullmatch(numeric_brand_pattern, str(x)) else x
)
usda_df['brand_name'] = usda_df['brand_name'].apply(
    lambda x: 'Unknown' if re.fullmatch(numeric_brand_pattern, str(x)) else x
)

# Also catch very long digit strings (barcodes embedded in names, e.g. '71682000008')
usda_df['brand_owner'] = usda_df['brand_owner'].apply(
    lambda x: 'Unknown' if re.search(r'^\d{5,}$', str(x).strip()) else x
)
usda_df['brand_name'] = usda_df['brand_name'].apply(
    lambda x: 'Unknown' if re.search(r'^\d{5,}$', str(x).strip()) else x
)

print('brand_owner remaining invalid:', usda_df['brand_owner'].str.fullmatch(r'[\d\s]+', na=False).sum())
print('brand_name remaining invalid:', usda_df['brand_name'].str.fullmatch(r'[\d\s]+', na=False).sum())


brand_owner remaining invalid: 0
brand_name remaining invalid: 0


In [15]:
# ── Fix brand values with stray brackets ──────────────────────────────
# e.g. [[Conagra Brands, Inc]] → Conagra Brands, Inc
# e.g. FREMONT [FISH MARKET]  → FREMONT FISH MARKET
usda_df['brand_owner'] = usda_df['brand_owner'].str.replace(r'\[\[|\]\]', '', regex=True).str.strip()
usda_df['brand_name']  = usda_df['brand_name'].str.replace(r'\[\[|\]\]', '', regex=True).str.strip()
usda_df['brand_owner'] = usda_df['brand_owner'].str.replace(r'\[|\]', '', regex=True).str.strip()
usda_df['brand_name']  = usda_df['brand_name'].str.replace(r'\[|\]', '', regex=True).str.strip()

print('brand_owner bracket issues remaining:',
      usda_df['brand_owner'].str.contains(r'\[', na=False, regex=True).sum())
print('brand_name  bracket issues remaining:',
      usda_df['brand_name'].str.contains(r'\[', na=False, regex=True).sum())


brand_owner bracket issues remaining: 0
brand_name  bracket issues remaining: 0


In [16]:
# ── Fix negative carbs ────────────────────────────────────────────────
print('Negative carbs before:', (usda_df['carbs_g'] < 0).sum())
usda_df.loc[usda_df['carbs_g'] < 0, 'carbs_g'] = 0
print('Negative carbs after:', (usda_df['carbs_g'] < 0).sum())


Negative carbs before: 10
Negative carbs after: 0


In [17]:
# ── Fix calorie outliers ──────────────────────────────────────────────
# Issue: ~2,865 rows have calories ≈ 4× what macros imply.
# Root cause: calories recorded per serving (≈240g) while macros are per 100g.
# Fix: recalculate calories from macros for affected rows.
# We do NOT simply clamp to a ceiling — that would artificially equalise
# very different foods (e.g. bacon grease ≠ apple).

usda_df['_expected_cal'] = usda_df['fat_g'] * 9 + usda_df['protein_g'] * 4 + usda_df['carbs_g'] * 4
usda_df['_cal_ratio']    = usda_df['calories'] / (usda_df['_expected_cal'] + 1)

# Rows with ratio consistently 3.5–4.5 and expected_cal > 10 are the per-serving error
cal_fix_mask = (
    (usda_df['_cal_ratio'] > 3.5) &
    (usda_df['_cal_ratio'] < 4.5) &
    (usda_df['_expected_cal'] > 10)
)

usda_df.loc[cal_fix_mask, 'calories'] = usda_df.loc[cal_fix_mask, '_expected_cal'].round(1)

# For remaining extreme outliers (ratio > 4.5) that are not pure fat,
# replace with grouped median — keeps values realistic without homogenising
extreme_cal_mask = (
    (usda_df['_cal_ratio'] > 4.5) &
    (usda_df['fat_g'] < 90)        # pure fat can legitimately be ~900 kcal/100g
)
usda_df.loc[extreme_cal_mask, 'calories'] = np.nan   # will be filled in the median step below

# Drop helper columns
usda_df.drop(columns=['_expected_cal', '_cal_ratio'], inplace=True)

print('Rows fixed (calorie ratio ~4):', cal_fix_mask.sum())
print('Rows set to NaN (extreme, non-fat):', extreme_cal_mask.sum())


Rows fixed (calorie ratio ~4): 2865
Rows set to NaN (extreme, non-fat): 156


In [18]:
# ── Fix sodium outliers ───────────────────────────────────────────────
# Sodium > 10,000 mg/100g is unrealistic for any food (table salt ≈ 38,758 mg/100g is the max).
# Values like 1,430,000 mg are clearly data-entry errors (off by 100–1000×).
# Strategy: for non-seasoning foods cap using grouped median; for seasoning/salt foods use 10,000 cap.

SEASONING_CATEGORIES = ['Spices and Herbs', 'Soups Sauces and Gravies', 'Condiments']
is_seasoning = usda_df['food_category'].isin(SEASONING_CATEGORIES)

# Regular foods: sodium > 10,000 mg → NaN (will be filled by grouped median)
usda_df.loc[~is_seasoning & (usda_df['sodium_mg'] > 10000), 'sodium_mg'] = np.nan

# Seasonings: sodium > 50,000 mg → NaN (salt itself is ~38,758 mg/100g)
usda_df.loc[is_seasoning & (usda_df['sodium_mg'] > 50000), 'sodium_mg'] = np.nan

print('Sodium outliers removed:', usda_df['sodium_mg'].isnull().sum())


Sodium outliers removed: 1040


In [19]:
# ── Fill all numeric nulls via grouped median ─────────────────────────
cols_to_fill = [
    'calories', 'carbs_g', 'calcium_mg', 'fat_g', 'protein_g',
    'saturated_fat_g', 'vitamin_c_mg', 'fiber_g', 'iron_mg',
    'sodium_mg', 'sugar_g', 'cholesterol_mg'
]

usda_df[cols_to_fill] = usda_df.groupby('food_type')[cols_to_fill].transform(
    lambda x: x.fillna(x.median())
)

# Global median fallback for any remaining NaN (tiny groups with all-null column)
usda_df[cols_to_fill] = usda_df[cols_to_fill].fillna(usda_df[cols_to_fill].median())

usda_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fdc_id           40000 non-null  int64  
 1   food_name        40000 non-null  str    
 2   data_type        40000 non-null  str    
 3   food_category    40000 non-null  str    
 4   brand_owner      40000 non-null  str    
 5   brand_name       40000 non-null  str    
 6   calories         40000 non-null  float64
 7   carbs_g          40000 non-null  float64
 8   calcium_mg       40000 non-null  float64
 9   fat_g            40000 non-null  float64
 10  protein_g        40000 non-null  float64
 11  saturated_fat_g  40000 non-null  float64
 12  vitamin_c_mg     40000 non-null  float64
 13  fiber_g          40000 non-null  float64
 14  iron_mg          40000 non-null  float64
 15  sodium_mg        40000 non-null  float64
 16  sugar_g          40000 non-null  float64
 17  cholesterol_mg   40000 

In [20]:
# Clean food_name: lowercase, remove punctuation, collapse whitespace
usda_df['food_name'] = (
    usda_df['food_name']
    .str.lower()
    .str.replace(r'"+', '', regex=True)
    .str.replace(r'[,/()-]', ' ', regex=True)
    .str.replace(r'[^a-z0-9\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Keep only rows that contain at least 2 consecutive letters
usda_df = usda_df[usda_df['food_name'].str.contains(r'[a-z]{2,}', na=False, regex=True)]

# [FIX] Also drop rows where food_name is fewer than 2 characters after cleaning
usda_df = usda_df[usda_df['food_name'].str.len() >= 2]
print('Rows after food_name cleaning:', len(usda_df))


Rows after food_name cleaning: 39997


In [21]:
# ── Final sanity check ────────────────────────────────────────────────
print('Null counts:')
print(usda_df.isnull().sum())
print()
print('Numeric describe:')
print(usda_df[cols_to_fill].describe())


Null counts:
fdc_id             0
food_name          0
data_type          0
food_category      0
brand_owner        0
brand_name         0
calories           0
carbs_g            0
calcium_mg         0
fat_g              0
protein_g          0
saturated_fat_g    0
vitamin_c_mg       0
fiber_g            0
iron_mg            0
sodium_mg          0
sugar_g            0
cholesterol_mg     0
health_score       0
food_type          0
dtype: int64

Numeric describe:
           calories       carbs_g     calcium_mg         fat_g     protein_g  \
count  39997.000000  39997.000000   39997.000000  39997.000000  39997.000000   
mean     252.949734     28.353270      94.371139     12.075162      8.248971   
std      281.630062     47.305135    1228.266683     19.085193     10.642463   
min        0.000000      0.000000       0.000000      0.000000      0.000000   
25%       80.000000      3.710000       8.000000      0.000000      0.460000   
50%      229.000000     15.800000      30.000000      4

In [22]:
# Save cleaned file
usda_df.to_csv('../data/silver_layer/usda_clean_data.csv', index=False)
print('Saved usda_clean_data.csv')


Saved usda_clean_data.csv


---
# Dataset 2: Healthy Foods
## Exploration


In [23]:
healthyFoods_df = pd.read_csv('../data/bronze_layer/healthyFoods.csv')


In [24]:
healthyFoods_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9028 entries, 0 to 9027
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   food_name     9028 non-null   str    
 1   food_type     9028 non-null   str    
 2   calories      9028 non-null   float64
 3   protein_g     9021 non-null   float64
 4   fat_g         9014 non-null   float64
 5   carbs_g       9002 non-null   float64
 6   fiber_g       8676 non-null   float64
 7   sugar_g       8027 non-null   float64
 8   sodium_mg     8956 non-null   float64
 9   health_score  9028 non-null   int64  
dtypes: float64(7), int64(1), str(2)
memory usage: 705.4 KB


In [25]:
healthyFoods_df.isnull().sum()

food_name          0
food_type          0
calories           0
protein_g          7
fat_g             14
carbs_g           26
fiber_g          352
sugar_g         1001
sodium_mg         72
health_score       0
dtype: int64

In [26]:
healthyFoods_df.describe()

,calories,protein_g,fat_g,carbs_g,fiber_g,sugar_g,sodium_mg,health_score
count,9028.000000,9021.000000,9014.000000,9002.000000,8676.000000,8027.000000,8956.000000,9028.000000
mean,329.047652,15.990428,9.948124,26.990668,4.926060,3.574429,390.662698,62.167147
std,316.877758,9.395426,13.400124,41.330864,9.373754,6.276089,11438.273033,4.004052
min,0.000000,0.000000,0.000000,-0.705000,0.000000,0.000000,0.000000,60.000000
25%,134.000000,10.600000,1.790000,0.000000,0.000000,0.000000,49.000000,60.000000
50%,247.000000,14.300000,5.220000,17.900000,3.600000,0.940000,119.000000,60.000000
75%,422.000000,20.900000,11.300000,50.000000,7.100000,4.350000,369.000000,60.000000
max,11800.000000,131.000000,150.000000,2940.000000,588.000000,100.000000,1000000.000000,75.000000


In [27]:
(usda_df['carbs_g'] < 0).sum()

np.int64(0)

In [28]:
pd.set_option('display.max_columns', None)
usda_df[usda_df['carbs_g'] < 0]

,fdc_id,food_name,data_type,food_category,brand_owner,brand_name,calories,carbs_g,calcium_mg,fat_g,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score,food_type


# Data Cleaning

**Steps:**
- Remove duplicate rows (run twice: before and after `food_name` normalisation, since cleaning can make previously distinct names identical)
- Replace negative `carbs_g` values with 0
- **Fix calorie outliers:** same macro-ratio correction as USDA (rows with `calories ≈ 4× macros` → recalculate; extreme outliers for non-fat foods → NaN)
- **Fix sodium outliers:** values > 10,000 mg/100g → set to NaN, filled by grouped median
- Fill nulls in numeric columns using median grouped by `food_type`; global median as fallback
- **Clean `food_name`:** lowercase, strip punctuation, collapse whitespace; drop rows with no real word content (< 2 letters)



In [29]:
print('Duplicates before:', healthyFoods_df.duplicated().sum())
healthyFoods_df.drop_duplicates(inplace=True)
print('Duplicates after:', healthyFoods_df.duplicated().sum())


Duplicates before: 559
Duplicates after: 0


In [30]:
# Fix negative carbs
healthyFoods_df.loc[healthyFoods_df['carbs_g'] < 0, 'carbs_g'] = 0


In [31]:
# ── Fix calorie outliers (same logic as USDA) ─────────────────────────
healthyFoods_df['_expected_cal'] = (
    healthyFoods_df['fat_g'] * 9 +
    healthyFoods_df['protein_g'] * 4 +
    healthyFoods_df['carbs_g'] * 4
)
healthyFoods_df['_cal_ratio'] = healthyFoods_df['calories'] / (healthyFoods_df['_expected_cal'] + 1)

hf_cal_fix_mask = (
    (healthyFoods_df['_cal_ratio'] > 3.5) &
    (healthyFoods_df['_cal_ratio'] < 4.5) &
    (healthyFoods_df['_expected_cal'] > 10)
)
healthyFoods_df.loc[hf_cal_fix_mask, 'calories'] = healthyFoods_df.loc[hf_cal_fix_mask, '_expected_cal'].round(1)

hf_extreme_cal = (
    (healthyFoods_df['_cal_ratio'] > 4.5) &
    (healthyFoods_df['fat_g'] < 90)
)
healthyFoods_df.loc[hf_extreme_cal, 'calories'] = np.nan

healthyFoods_df.drop(columns=['_expected_cal', '_cal_ratio'], inplace=True)
print('Healthy Foods calorie rows fixed (ratio ~4):', hf_cal_fix_mask.sum())
print('Healthy Foods extreme outliers set to NaN:', hf_extreme_cal.sum())


Healthy Foods calorie rows fixed (ratio ~4): 985
Healthy Foods extreme outliers set to NaN: 3


In [32]:
# ── Fix sodium outliers ───────────────────────────────────────────────
if 'sodium_mg' in healthyFoods_df.columns:
    healthyFoods_df.loc[healthyFoods_df['sodium_mg'] > 10000, 'sodium_mg'] = np.nan


In [33]:
# ── Fill nulls via grouped median ─────────────────────────────────────
hf_numeric = ['calories', 'carbs_g', 'fat_g', 'protein_g', 'fiber_g', 'sugar_g', 'sodium_mg']
hf_numeric = [c for c in hf_numeric if c in healthyFoods_df.columns]

healthyFoods_df[hf_numeric] = healthyFoods_df.groupby('food_type')[hf_numeric].transform(
    lambda x: x.fillna(x.median())
)
healthyFoods_df[hf_numeric] = healthyFoods_df[hf_numeric].fillna(healthyFoods_df[hf_numeric].median())

healthyFoods_df.info()


<class 'pandas.DataFrame'>
Index: 8469 entries, 0 to 9027
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   food_name     8469 non-null   str    
 1   food_type     8469 non-null   str    
 2   calories      8469 non-null   float64
 3   protein_g     8469 non-null   float64
 4   fat_g         8469 non-null   float64
 5   carbs_g       8469 non-null   float64
 6   fiber_g       8469 non-null   float64
 7   sugar_g       8469 non-null   float64
 8   sodium_mg     8469 non-null   float64
 9   health_score  8469 non-null   int64  
dtypes: float64(7), int64(1), str(2)
memory usage: 727.8 KB


In [34]:
# Clean food_name: lowercase, remove punctuation, collapse whitespace
healthyFoods_df['food_name'] = (
    healthyFoods_df['food_name']
    .str.lower()
    .str.replace(r'"+', '', regex=True)
    .str.replace(r'[,/()-]', ' ', regex=True)
    .str.replace(r'[^a-z0-9\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
healthyFoods_df = healthyFoods_df[healthyFoods_df['food_name'].str.contains(r'[a-z]{2,}', na=False, regex=True)]
healthyFoods_df = healthyFoods_df[healthyFoods_df['food_name'].str.len() >= 2]
print('Rows after food_name cleaning:', len(healthyFoods_df))


Rows after food_name cleaning: 8468


In [35]:
# [FIX] drop_duplicates AFTER food_name cleaning
# (names that were different originally may become identical after normalization)
before = len(healthyFoods_df)
healthyFoods_df.drop_duplicates(inplace=True)
print(f'Dropped {before - len(healthyFoods_df)} duplicates')
print('Remaining duplicates:', healthyFoods_df.duplicated().sum())


Dropped 41 duplicates
Remaining duplicates: 0


In [36]:
# Save cleaned file
healthyFoods_df.to_csv('../data/silver_layer/healthyFoods_clean_data.csv', index=False)
print('Saved healthyFoods_clean_data.csv')

Saved healthyFoods_clean_data.csv


---
# Dataset 3: Allergens
## Exploration


In [37]:
allergns_df = pd.read_csv('../data/bronze_layer/allergens.csv', encoding='utf-8')

In [38]:
allergns_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        4785 non-null   str    
 1   brands              4751 non-null   str    
 2   categories          4913 non-null   str    
 3   ingredients         4797 non-null   str    
 4   nutriscore_grade    4983 non-null   str    
 5   nova_group          4522 non-null   float64
 6   ecoscore_grade      4984 non-null   str    
 7   allergens           3332 non-null   str    
 8   energy_kcal         4682 non-null   float64
 9   fat_100g            4695 non-null   float64
 10  saturated_fat_100g  4617 non-null   float64
 11  carbs_100g          4676 non-null   float64
 12  sugars_100g         4631 non-null   float64
 13  fiber_100g          3495 non-null   float64
 14  proteins_100g       4690 non-null   float64
 15  salt_100g           4653 non-null   float64
 16  sodium_100g      

In [39]:
allergns_df.isnull().sum()

product_name           212
brands                 246
categories              84
ingredients            200
nutriscore_grade        14
nova_group             475
ecoscore_grade          13
allergens             1665
energy_kcal            315
fat_100g               302
saturated_fat_100g     380
carbs_100g             321
sugars_100g            366
fiber_100g            1502
proteins_100g          307
salt_100g              344
sodium_100g            344
contains_gluten          0
contains_dairy           0
contains_nuts            0
contains_soy             0
contains_eggs            0
contains_fish            0
food_type                0
dtype: int64

## Data Cleaning

**Steps:**
- Remove duplicate rows (also re-run after text standardisation)
- Replace sentinel category values (`NaN`, `null`, `unknown`, etc.) with proper NaN
- Infer missing `categories` from allergen boolean flags and keyword matching in `product_name`, `ingredients`, and `brands`
- Fix `product_name` values that are barcodes, scientific notation numbers, or date strings → replace with `brands` value or `'Unknown Product'`
- Fill missing `brands` from the first word of `product_name`; fallback to `'Unknown'`; strip trailing commas; replace pure short-numeric brand values with `'Unknown'`
- **Fix allergens comprehensively:** normalise sentinel values; parse existing allergen strings; extract allergens from `ingredients` text; cross-validate against boolean columns (`contains_gluten`, `contains_dairy`, etc.); union all sources; apply category-level fallback for still-empty rows; strip `en:`/`ar:` language prefixes; deduplicate and sort; replace `'none'` with `'None'`
- Drop unused columns: `ingredients`, `food_type`, `ecoscore_grade`
- Split `categories` into `category` (first segment) and `subcategory` (last segment); drop original `categories` column
- Translate non-English text columns to English using `GoogleTranslator`
- Standardise text columns (`product_name`, `brands`, `category`, `subcategory`): lowercase → remove standalone digits/punctuation → Title Case
- **Fix `nutriscore_grade`:** uppercase and strip; infer from `nova_group` where missing; fill remaining with `'UNKNOWN'`
- Fill missing `nova_group` with `0`; cast to int
- Remove physiologically impossible numeric outliers (`fat_100g` > 100, `carbs_100g` > 100, etc.); fill remaining nulls via median grouped by `category`; global median fallback
- Fill remaining `subcategory` nulls with `'Unknown'`



In [40]:
# ── Helper functions ──────────────────────────────────────────────────

def split_categories(text):
    if pd.isna(text):
        return pd.Series([np.nan, np.nan])
    parts = [p.strip() for p in str(text).split(',')]
    parts = [p.split(':', 1)[1].strip() if ':' in p else p for p in parts]
    parts = [p for p in parts if p]
    left  = parts[0]  if len(parts) > 0 else np.nan
    right = parts[-1] if len(parts) > 1 else np.nan
    return pd.Series([left, right])


def clean_allergens_text(text):
    """Strip language prefixes, deduplicate, sort. Returns clean comma-separated string."""
    if pd.isna(text):
        return np.nan
    parts = str(text).split(',')
    cleaned = [
        re.sub(r'^(en|ar|fr|es|de|it|nl|pt|pl):', '', p.strip()).strip()
        for p in parts
    ]
    cleaned = [p for p in cleaned if p and p.lower() not in ('nan', 'null', '')]  
    cleaned = sorted(set(cleaned))  # deduplicate & sort for consistency
    return ', '.join(cleaned) if cleaned else np.nan


# ── Translation helpers ───────────────────────────────────────────────
translator = GoogleTranslator(source='auto', target='en')

def contains_non_english(text):
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

def translate_batch(batch):
    try:
        return translator.translate_batch(batch)
    except Exception:
        result = []
        for t in batch:
            try:    result.append(translator.translate(t))
            except: result.append(t)
        return result

def translate_column_fast(series):
    cleaned     = series.dropna().astype(str).str.strip()
    unique_vals = cleaned.unique()
    to_translate = [t for t in unique_vals if contains_non_english(t)]
    if not to_translate:
        return series
    batches  = [to_translate[i:i+50] for i in range(0, len(to_translate), 50)]
    trans_map = {}
    with ThreadPoolExecutor(max_workers=5) as ex:
        for batch, translated in zip(batches, ex.map(translate_batch, batches)):
            trans_map.update(dict(zip(batch, translated)))
    for t in unique_vals:
        if t not in trans_map:
            trans_map[t] = t
    return series.map(lambda x: trans_map.get(str(x).strip(), x) if pd.notna(x) else x)


# ── Standardise text ──────────────────────────────────────────────────
def standardize_text(text):
    if pd.isna(text):
        return np.nan
    original = str(text).strip()
    result   = original.lower()
    result   = result.replace('-', ' ')
    result   = re.sub(r'\b\d+\b', '', result)       # remove standalone digits
    result   = re.sub(r'[^a-z\s,]', '', result)
    result   = re.sub(r'\s+', ' ', result).strip()
    result   = result.title()
    return result if result else original


print('Functions defined.')


Functions defined.


In [41]:
# ── Clean categories sentinel values ──────────────────────────────────
allergns_df['categories'] = allergns_df['categories'].replace(
    ['NaN', 'nan', 'NULL', 'null', '', 'unknown', 'UNKNOWN'], np.nan
)


# ── Infer missing categories ──────────────────────────────────────────
def infer_category(row):
    if pd.notnull(row['categories']):
        return row['categories']

    text = ' '.join([
        str(row.get('product_name', '')),
        str(row.get('ingredients', '')),
        str(row.get('brands', ''))
    ]).lower()

    gluten = bool(row.get('contains_gluten', False))
    dairy  = bool(row.get('contains_dairy', False))
    nuts   = bool(row.get('contains_nuts', False))
    soy    = bool(row.get('contains_soy', False))
    eggs   = bool(row.get('contains_eggs', False))
    fish   = bool(row.get('contains_fish', False))

    beverage_kw = ['water','juice','drink','beverage','cola','soda','tea','coffee','milkshake','smoothie','مياه','عصير','مشروب']
    dairy_kw    = ['milk','yogurt','cheese','cream','butter']
    snack_kw    = ['biscuit','cookie','cake','wafer','cracker','snack','chocolate','bar','croissant','muffin','donut']
    cereal_kw   = ['bread','toast','pasta','rice','oats','corn','cereal','flour','wheat']
    seafood_kw  = ['fish','tuna','salmon','sardine','shrimp']

    if fish or any(k in text for k in seafood_kw):
        return 'en:seafood, en:fishes-and-their-products, en:canned-foods'
    if any(k in text for k in beverage_kw):
        return 'en:beverages-and-beverages-preparations, en:beverages, en:waters'
    if dairy and any(k in text for k in dairy_kw):
        return 'en:dairies, en:fermented-foods, en:fermented-milk-products'
    if (gluten or eggs or dairy or nuts) and any(k in text for k in snack_kw):
        return 'en:snacks, en:sweet-snacks, en:biscuits-and-cakes'
    if gluten or any(k in text for k in cereal_kw):
        return 'en:plant-based-foods-and-beverages, en:plant-based-foods, en:cereals-and-potatoes'
    if soy:
        return 'en:plant-based-foods-and-beverages, en:plant-based-foods, en:meals'
    return 'en:plant-based-foods-and-beverages, en:plant-based-foods, en:fruits-and-vegetables-based-foods'


allergns_df['categories'] = allergns_df.apply(infer_category, axis=1)
print('Remaining null categories:', allergns_df['categories'].isnull().sum())
print('Top categories:')
print(allergns_df['categories'].value_counts().head(8))


Remaining null categories: 0
Top categories:
categories
en:plant-based-foods-and-beverages, en:plant-based-foods, en:cereals-and-potatoes                 498
en:dairies, en:fermented-foods, en:fermented-milk-products                                        473
en:plant-based-foods-and-beverages, en:plant-based-foods, en:breakfasts                           351
en:snacks, en:sweet-snacks, en:biscuits-and-cakes                                                 333
en:plant-based-foods-and-beverages, en:plant-based-foods, en:fruits-and-vegetables-based-foods    280
en:beverages-and-beverages-preparations, en:plant-based-foods-and-beverages, en:beverages         223
en:snacks, en:sweet-snacks, en:cocoa-and-its-products                                             219
en:plant-based-foods-and-beverages, en:plant-based-foods, en:snacks                               168
Name: count, dtype: int64


In [42]:
# ── Fix product_name ──────────────────────────────────────────────────
# 1. Replace numeric/barcode product_name (e.g. '6.11E+12', '41022') with brand name or Unknown
def is_barcode_name(text):
    """Return True if the product name looks like a barcode/scientific notation number."""
    if pd.isna(text):
        return False
    t = str(text).strip()
    # Pure numeric, scientific notation (e.g. 6.11E+12), or very short all-digit strings
    return bool(re.fullmatch(r'[\d\.E\+\-]+', t))

def fill_product_name(row):
    pn = row['product_name']
    if pd.isna(pn) or is_barcode_name(pn):
        if pd.notnull(row.get('brands')) and not is_barcode_name(row.get('brands')):
            return row['brands']
        if pd.notnull(row.get('ingredients')):
            first = str(row['ingredients']).split(',')[0].strip()
            if first and not is_barcode_name(first):
                return first
        return 'Unknown Product'
    return pn

allergns_df['product_name'] = allergns_df.apply(fill_product_name, axis=1)

# Also strip numeric-only suffixes like '14 Maxi' → keep as-is (they have real words)
# Only pure numeric names are replaced above
print('Remaining numeric product names:', allergns_df['product_name'].apply(is_barcode_name).sum())

Remaining numeric product names: 0


In [43]:
# ── Fix product_name that is a date (e.g. '11/12/2024') ──────────────
date_mask = allergns_df['product_name'].str.match(r'^\d{1,2}/\d{1,2}/\d{4}$', na=False)
allergns_df.loc[date_mask, 'product_name'] = (
    allergns_df.loc[date_mask, 'brands']
    .where(allergns_df.loc[date_mask, 'brands'].notna(), other='Unknown Product')
)
print('Date product_names fixed:', date_mask.sum())

# ── Fix brands that are pure short numbers (e.g. '03') ───────────────
num_brand_mask = allergns_df['brands'].str.match(r'^\d{1,4}$', na=False)
allergns_df.loc[num_brand_mask, 'brands'] = 'Unknown'
print('Pure-numeric brand values fixed:', num_brand_mask.sum())


Date product_names fixed: 1
Pure-numeric brand values fixed: 0


In [44]:
# ── Fix brands ────────────────────────────────────────────────────────
allergns_df['brands'] = allergns_df['brands'].replace(['NaN', 'null', 'NULL', ''], np.nan)

# Fill missing brands using first word of product_name
allergns_df['brands'] = allergns_df['brands'].fillna(
    allergns_df['product_name'].str.split().str[0]
)
allergns_df['brands'] = allergns_df['brands'].fillna('Unknown')

# Strip trailing commas
allergns_df['brands'] = allergns_df['brands'].str.strip().str.rstrip(',').str.strip()


In [45]:
# ── Fix allergens – comprehensive approach ────────────────────────────
# Keyword → allergen mapping
allergen_map = {
    'milk': 'milk', 'lait': 'milk', 'dairy': 'milk', 'cream': 'milk',
    'butter': 'milk', 'cheese': 'milk', 'yogurt': 'milk',
    'wheat': 'gluten', 'blé': 'gluten', 'gluten': 'gluten', 'barley': 'gluten',
    'rye': 'gluten', 'oat': 'gluten',
    'soy': 'soybeans', 'soja': 'soybeans',
    'peanut': 'peanuts', 'groundnut': 'peanuts',
    'nut': 'nuts', 'almond': 'nuts', 'cashew': 'nuts', 'walnut': 'nuts',
    'hazelnut': 'nuts', 'pecan': 'nuts', 'pistachio': 'nuts',
    'fish': 'fish', 'poisson': 'fish', 'tuna': 'fish', 'salmon': 'fish',
    'shrimp': 'crustaceans', 'prawn': 'crustaceans', 'crab': 'crustaceans',
    'egg': 'eggs', 'oeuf': 'eggs', 'œuf': 'eggs',
    'sesame': 'sesame-seeds',
    'celery': 'celery', 'mustard': 'mustard', 'sulphite': 'sulphites',
    'sulfite': 'sulphites', 'lupin': 'lupin', 'mollusc': 'molluscs',
}

def extract_allergens_from_text(text):
    if pd.isna(text):
        return set()
    text = str(text).lower()
    found = set()
    for kw, allergen in allergen_map.items():
        if kw in text:
            found.add(allergen)
    return found


def build_allergen_from_booleans(row):
    """Build allergen set from the boolean columns."""
    found = set()
    if row.get('contains_gluten'):  found.add('gluten')
    if row.get('contains_dairy'):   found.add('milk')
    if row.get('contains_nuts'):    found.add('nuts')
    if row.get('contains_soy'):     found.add('soybeans')
    if row.get('contains_eggs'):    found.add('eggs')
    if row.get('contains_fish'):    found.add('fish')
    return found


# Step 1: normalise sentinel values
allergns_df['allergens'] = allergns_df['allergens'].replace(
    ['NaN', 'nan', 'NULL', 'null', ''], np.nan
)

# Step 2: parse existing allergens string into a set
def parse_allergen_str(text):
    if pd.isna(text):
        return set()
    parts = re.split(r'[,;]', str(text))
    cleaned = set()
    for p in parts:
        p = re.sub(r'^(en|ar|fr|es|de|it):', '', p.strip()).strip().lower()
        if p and p not in ('nan', 'null', 'none', 'unknown'):
            cleaned.add(p)
    return cleaned


# Step 3: combine allergens from all sources (existing text + ingredients + booleans)
def merge_allergens(row):
    base     = parse_allergen_str(row['allergens'])
    from_ing = extract_allergens_from_text(row.get('ingredients', ''))
    from_bool = build_allergen_from_booleans(row)
    # Union all sources
    merged = base | from_ing | from_bool
    if not merged:
        # Category-level fallback
        cat = str(row.get('categories', '')).lower()
        if 'dair' in cat or 'milk' in cat:   merged.add('milk')
        if 'fish' in cat or 'seafood' in cat: merged.add('fish')
        if 'nuts' in cat or 'cocoa' in cat:  merged.add('nuts')
    return ', '.join(sorted(merged)) if merged else 'unknown'


allergns_df['allergens'] = allergns_df.apply(merge_allergens, axis=1)

# Step 4: clean the result (strip language prefixes, deduplicate)
allergns_df['allergens'] = allergns_df['allergens'].apply(clean_allergens_text)

# Step 5: replace 'none' string (from 'en:none') with the literal 'None'
allergns_df['allergens'] = allergns_df['allergens'].apply(
    lambda x: 'None' if str(x).strip().lower() == 'none' else x
)

print('Allergens value counts (top 10):')
print(allergns_df['allergens'].value_counts().head(10))

Allergens value counts (top 10):
allergens
unknown                         1251
milk                             799
gluten                           555
nuts                             189
gluten, soybeans                 156
soybeans                         144
gluten, milk                     135
gluten, milk, soybeans           131
eggs, gluten, milk, soybeans      97
nuts, peanuts                     81
Name: count, dtype: int64


In [46]:
# ── Drop unused columns ───────────────────────────────────────────────
allergns_df = allergns_df.drop(columns=['ingredients', 'food_type', 'ecoscore_grade'])
allergns_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        4997 non-null   str    
 1   brands              4997 non-null   str    
 2   categories          4997 non-null   str    
 3   nutriscore_grade    4983 non-null   str    
 4   nova_group          4522 non-null   float64
 5   allergens           4997 non-null   str    
 6   energy_kcal         4682 non-null   float64
 7   fat_100g            4695 non-null   float64
 8   saturated_fat_100g  4617 non-null   float64
 9   carbs_100g          4676 non-null   float64
 10  sugars_100g         4631 non-null   float64
 11  fiber_100g          3495 non-null   float64
 12  proteins_100g       4690 non-null   float64
 13  salt_100g           4653 non-null   float64
 14  sodium_100g         4653 non-null   float64
 15  contains_gluten     4997 non-null   bool   
 16  contains_dairy   

In [47]:
# ── Split categories ──────────────────────────────────────────────────
allergns_df[['category', 'subcategory']] = allergns_df['categories'].apply(split_categories)
allergns_df = allergns_df.drop(columns=['categories'])
print(allergns_df[['category', 'subcategory']].head(5))

                               category              subcategory
0  beverages-and-beverages-preparations                   waters
1                               dairies  fermented-milk-products
2  beverages-and-beverages-preparations                   waters
3  beverages-and-beverages-preparations                   waters
4  beverages-and-beverages-preparations                   waters


In [48]:
# ── Translate non-English columns ─────────────────────────────────────
for col in allergns_df.select_dtypes(include=['string', 'object']).columns:
    if col in ('allergens', 'nutriscore_grade'):   # no point translating; already English tags
        continue
    print(f'  Translating: {col}')
    allergns_df[col] = translate_column_fast(allergns_df[col])
print('Translation done.')

  Translating: product_name
  Translating: brands
  Translating: category
  Translating: subcategory
Translation done.


In [49]:
# ── Standardise text columns ──────────────────────────────────────────
text_cols_to_standardize = ['product_name', 'brands', 'category', 'subcategory']
for col in text_cols_to_standardize:
    if col in allergns_df.columns:
        allergns_df[col] = allergns_df[col].apply(standardize_text)

# Known brand fix
allergns_df['brands'] = allergns_df['brands'].replace('Sir Ali', 'Sidi Ali')

print('Standardisation done.')
allergns_df.head(3)

Standardisation done.


,product_name,brands,nutriscore_grade,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,category,subcategory
0,Sidi Ali,Sidi Ali,A,NaN,unknown,0.0,0.0,0.0,4.2,1.4,0.0,0.0,0.000,0.000,False,False,False,False,False,False,Beverages And Beverages Preparations,Waters
1,Perly,Perly,UNKNOWN,3.0,"banana, milk",97.0,3.0,NaN,9.4,NaN,NaN,8.0,NaN,NaN,False,True,False,False,False,False,Dairies,Fermented Milk Products
2,Sidi Ali,Sidi Ali,A,1.0,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065,0.026,False,False,False,False,False,False,Beverages And Beverages Preparations,Waters


In [50]:
# ── Fix nutriscore_grade ──────────────────────────────────────────────
# Standardise to uppercase; keep valid grades A-E, UNKNOWN, NOT-APPLICABLE
allergns_df['nutriscore_grade'] = allergns_df['nutriscore_grade'].str.upper().str.strip()

# Map nova_group → nutriscore for rows still missing a grade
nova_to_nutriscore = {1.0: 'A', 2.0: 'B', 3.0: 'C', 4.0: 'D'}
allergns_df['nutriscore_grade'] = allergns_df['nutriscore_grade'].fillna(
    allergns_df['nova_group'].map(nova_to_nutriscore)
)
allergns_df['nutriscore_grade'] = allergns_df['nutriscore_grade'].fillna('UNKNOWN')

print('nutriscore_grade distribution:')
print(allergns_df['nutriscore_grade'].value_counts())

nutriscore_grade distribution:
nutriscore_grade
C                 1072
E                  995
D                  899
A                  833
B                  648
UNKNOWN            415
NOT-APPLICABLE     135
Name: count, dtype: int64


In [51]:
# ── Fill nova_group ───────────────────────────────────────────────────
allergns_df['nova_group'] = allergns_df['nova_group'].fillna(0).astype(int)

In [52]:
# ── Fill numeric columns ──────────────────────────────────────────────
allerg_numeric = [
    'energy_kcal', 'fat_100g', 'saturated_fat_100g', 'carbs_100g',
    'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g'
]

# Remove physiologically impossible outliers before filling
# (energy_kcal > 900 for non-fat foods is unlikely; fat_100g > 100 is impossible)
allergns_df.loc[allergns_df['fat_100g'] > 100, 'fat_100g'] = np.nan
allergns_df.loc[allergns_df['saturated_fat_100g'] > 100, 'saturated_fat_100g'] = np.nan
allergns_df.loc[allergns_df['carbs_100g'] > 100, 'carbs_100g'] = np.nan
allergns_df.loc[allergns_df['proteins_100g'] > 100, 'proteins_100g'] = np.nan
# energy_kcal > 900 is only valid for pure fats
# For non-fat-dominant products, cap at 900
fat_dominant_mask = allergns_df['fat_100g'].fillna(0) >= 80
allergns_df.loc[~fat_dominant_mask & (allergns_df['energy_kcal'] > 900), 'energy_kcal'] = np.nan

# Fill via grouped median
allergns_df[allerg_numeric] = allergns_df.groupby('category')[allerg_numeric].transform(
    lambda x: x.fillna(x.median())
)
# Global median fallback
allergns_df[allerg_numeric] = allergns_df[allerg_numeric].fillna(
    allergns_df[allerg_numeric].median()
)
print('Numeric nulls after fill:')
print(allergns_df[allerg_numeric].isnull().sum())

Numeric nulls after fill:
energy_kcal           0
fat_100g              0
saturated_fat_100g    0
carbs_100g            0
sugars_100g           0
fiber_100g            0
proteins_100g         0
salt_100g             0
sodium_100g           0
dtype: int64


In [53]:
# ── Remove duplicates after standardisation ───────────────────────────
before = len(allergns_df)
allergns_df.drop_duplicates(inplace=True)
print(f'Dropped {before - len(allergns_df)} duplicates. Remaining: {len(allergns_df)}')

Dropped 71 duplicates. Remaining: 4926


In [54]:
# ── Fill remaining subcategory nulls ──────────────────────────────────
allergns_df['subcategory'] = allergns_df['subcategory'].fillna('Unknown')

print('Final null counts:')
print(allergns_df.isnull().sum())

Final null counts:
product_name          0
brands                0
nutriscore_grade      0
nova_group            0
allergens             0
energy_kcal           0
fat_100g              0
saturated_fat_100g    0
carbs_100g            0
sugars_100g           0
fiber_100g            0
proteins_100g         0
salt_100g             0
sodium_100g           0
contains_gluten       0
contains_dairy        0
contains_nuts         0
contains_soy          0
contains_eggs         0
contains_fish         0
category              0
subcategory           0
dtype: int64


In [55]:
# Save cleaned file
allergns_df.to_csv('../data/silver_layer/allergens_clean_data.csv', index=False)
print('Saved allergens_clean_data.csv')

Saved allergens_clean_data.csv
